In [25]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [26]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

## LER OS RETORNOS, SCORE e SELIC

In [27]:
# Retornos

dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    nome = f'df_ativos_{ano_usado}.csv'
    caminho = Path('retornos_ativos_otimizacao') / nome
    dfs[ano_usado] = pd.read_csv(caminho).set_index('date')
    dfs[ano_usado] = dfs[ano_usado].drop(columns=dfs[ano_usado].columns[(dfs[ano_usado] == 0).all()])
    print(ano,'-',len(dfs[ano_usado].columns))
    

2025-12-31 - 78
2024-12-31 - 78
2023-12-31 - 78
2022-12-31 - 77
2021-12-31 - 77
2020-12-31 - 73
2019-12-31 - 73
2018-12-31 - 70
2017-12-31 - 70
2016-12-31 - 67
2015-12-31 - 62


In [28]:
## SELIC para Sigma e excesso
selic_d = pd.read_csv('selic/selic_diario.csv').set_index('date')

## EXCESSO DOS ANOS e SIGMA (MAtriz de covariancia)

##### EXCESSO PARA TODOS

In [29]:
dict_excesso = {}
for an in anos:
    ano = an.split("-")[0]

    try:
        print(f"=============== \n EXCESSO {ano}\n ============")
        print("Atualização, Ano: ",ano)
        df = dfs[ano]
        # df_f = pd.DataFrame(eval(df))
        df_f = df.copy()
        slc = selic_d[selic_d.index.isin(df_f.index)]
        slc['valor_diario'] = slc['valor_diario']/100

        print(f"Tamanho DF de {ano}: ", len(df_f))
        print(f"Tamanho Selic: ", len(slc['valor_diario']))
        ano = int(ano)
        dict_excesso[ano] = df_f.sub(slc['valor_diario'],axis=0)
        print("tamanho final do Excesso: ",len(dict_excesso[ano]))

        print(f"============\n SIGMA {ano}\n===========")            

    except Exception as e:
        print(e)
        print("ERror")

 EXCESSO 2025
Atualização, Ano:  2025
Tamanho DF de 2025:  123
Tamanho Selic:  123
tamanho final do Excesso:  123
 SIGMA 2025
 EXCESSO 2024
Atualização, Ano:  2024
Tamanho DF de 2024:  122
Tamanho Selic:  122
tamanho final do Excesso:  122
 SIGMA 2024
 EXCESSO 2023
Atualização, Ano:  2023
Tamanho DF de 2023:  121
Tamanho Selic:  121
tamanho final do Excesso:  121
 SIGMA 2023
 EXCESSO 2022
Atualização, Ano:  2022
Tamanho DF de 2022:  124
Tamanho Selic:  124
tamanho final do Excesso:  124
 SIGMA 2022
 EXCESSO 2021
Atualização, Ano:  2021
Tamanho DF de 2021:  123
Tamanho Selic:  123
tamanho final do Excesso:  123
 SIGMA 2021
 EXCESSO 2020
Atualização, Ano:  2020
Tamanho DF de 2020:  120
Tamanho Selic:  120
tamanho final do Excesso:  120
 SIGMA 2020
 EXCESSO 2019
Atualização, Ano:  2019
Tamanho DF de 2019:  123
Tamanho Selic:  123
tamanho final do Excesso:  123
 SIGMA 2019
 EXCESSO 2018
Atualização, Ano:  2018
Tamanho DF de 2018:  119
Tamanho Selic:  119
tamanho final do Excesso:  119
 SIG

## Hiperparâmetros

In [30]:
vb_cardinalidade_max = 10
vb_cardinalidade_min = 10
vb_peso_maximo = 0.20
vb_peso_minimo = 0.02
vb_theta = 0.5

### Fazendo otimização ano a ano e comparando com o proximo ano

In [31]:
anos = ['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']
print(anos)

['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']


In [32]:
y = []
carteiras_anuais = {}
melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])
print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)
lista_ativos_finais = {}
for an in anos:
    ano = an.split("-")[0]
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass
    try:

            #SCORE ------------

        score_todos_anos = pd.read_csv(f'score_mf/{ano}/mf_{ano}.csv').set_index('ano')
        score_usado = score_todos_anos.dropna(axis=1).copy()
        # print(score_usado)
        print("Score Atualizado")
        ano_um = int(ano)

            # RETORNO ---------------

        # retorno_usado = pd.DataFrame(eval(df_usado))
        df_usado = dfs[str(ano_um)]
        retorno_usado = df_usado.copy()
        retorno_usado = retorno_usado[score_usado.columns]

            # Lista ativos por ano ---------
        lista_ativos_finais[ano_um] = score_usado.columns.tolist()
        print("Retornos atualizados")

            # EXCESSO retorno - rf ------------

        excesso_usado = dict_excesso[ano_um][score_usado.columns]
        print("Excessos Atualizados")

            # SIGMA ---------------

        sigma_usado = excesso_usado.cov()
        print("Sigmas Atualizados")
        print("-----")
    except Exception as e:
        print("ERRRRRRRRRRRROR")
        print(e)
    # if df_usado == 'df_ativos_2015':
    #     continue
    # else:
    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",int(ano)+1)
    print("## UTILIZANDO SCORE DO ANO DE: ",ano)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",str(ano_um))
    print("## UTILIZANDO EXCESSO DO ANO DE: ",ano_um)
    print("## UTILIZANDO SIGMAS DO ANO DE: ",ano_um)
    print(len(retorno_usado.columns))
    print(len(score_usado.columns))
    print(len(sigma_usado.columns))
    print(len(excesso_usado.columns))

    model = pyo.ConcreteModel()
    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = retorno_usado.columns)
    model.ativos = pyo.RangeSet(0, len(retorno_usado.columns)-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    model.theta = pyo.Param(initialize=vb_theta)
    model.score = pyo.Param(model.ativos, initialize=lambda model,a: score_usado.iloc[0,a])
    model.cardinalidade_valor_max = pyo.Param(initialize=vb_cardinalidade_max)
    model.cardinalidade_valor_min = pyo.Param(initialize=vb_cardinalidade_min)
    model.peso_maximo = pyo.Param(initialize=vb_peso_maximo)
    model.peso_minimo = pyo.Param(initialize=vb_peso_minimo)
    model.x = pyo.Var(model.ativos,bounds=(0,1),domain=pyo.NonNegativeReals)
    model.y = pyo.Var(model.ativos, within=pyo.Binary)
    model.excesso = pyo.Param( model.ativos , initialize = lambda model,a: excesso_usado.mean().iloc[a])
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize = lambda model,a,b: sigma_usado.iloc[a,b])
    model.s = pyo.Param(initialize = 2, mutable=True)
    model.r = pyo.Var(within=pyo.NonNegativeReals,bounds=(0,10))
    #-------------------------------------- FUNÇÕES
    #=============================
    # Função Objetivo
    #=============================
    def func_objetivo_1(model):
        retorno_esperado = model.theta * sum(
            model.retornos_ativos[dia, a] * model.x[a] for a in model.ativos for dia in model.dias
        )
        # retorno_esperado = model.theta * model.r
        # A dúvida de qual retorno usar (Retorno BRUTO ou Excesso (retorno - selic))

        score_total = (1 - model.theta) * sum(
            model.x[a] * model.score[a] for a in model.ativos
        )
        return retorno_esperado + score_total
    model.obj1 = pyo.Objective(rule=func_objetivo_1, sense=pyo.maximize)
    #=============================
    # RESTRIÇÕES
    #=============================
    def def_r(model):
        return model.r == sum(model.excesso[a]*model.x[a]for a in model.ativos)
    model.const_def_r = pyo.Constraint(rule=def_r)
    ## modelos de Programação de Cone de Segunda Ordem (SOCP) 
    ## e Programação Quadrática com Restrições (QCP)
    def cone(model):
        return model.r**2 >= model.s**2 * sum(model.x[a]*model.sigma[a,b]*model.x[b] for a in model.ativos for b in model.ativos)
    model.constr_cone = pyo.Constraint(rule=cone)
    #REstricao 1 x só ativa se y = 1
    def restr_vinculo_x_y(model, a):
        return model.x[a] <= model.y[a]
    model.const_restr_vinculo_x_y = pyo.Constraint(model.ativos, rule=restr_vinculo_x_y)
    #peso maximo por acao
    def rule_peso_maximo(model, a):
        # return model.x[a] <= 1/model.cardinalidade_valor
        return model.x[a] <= model.peso_maximo * model.y[a]
    model.const_peso_maximo = pyo.Constraint(model.ativos, rule=rule_peso_maximo)
    #peso minimo por acao
    def rule_peso_minimo(model, a):
        return model.x[a] >= model.peso_minimo * model.y[a]  # se y=1, então x >= 0.05
    model.const_peso_minimo = pyo.Constraint(model.ativos, rule=rule_peso_minimo)
    #Restrição 2 soma peso 1
    def soma_peso_1(model):
        return sum(model.x[a] for a in model.ativos) == 1
    model.const_soma_peso_1 = pyo.Constraint(rule=soma_peso_1)
    def cardinalidade_min(model):
        return sum(
            model.y[a] for a in model.ativos
            ) >= model.cardinalidade_valor_min
    model.const_cardinalidade_total_min = pyo.Constraint(rule=cardinalidade_min)
    def cardinalidade_max(model):
        return sum(
            model.y[a] for a in model.ativos
            ) <= model.cardinalidade_valor_max
    model.const_cardinalidade_total_max = pyo.Constraint(rule=cardinalidade_max)
    # NOTEBOOOK
    # opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')
    # PC
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    s_lo = 0.01        # <-- o Sharpe DIÁRIO 
    s_hi = 1   # teto: 2x o melhor ativo individual
    tol  = 0.001
    print("MODEL.S.VALUE => ",model.s.value)
    melhor_pesos = None
    historico = []
    carteiras_criadas = []
    # print(f"Valor a ser batido com S_LO: {s_lo} e S_HI: {s_hi} ... Diferença: {s_hi-s_lo}")
    print(f"Começando o WHILE do ano {ano}")
    while s_hi - s_lo > tol:
        s_a_ser_usado = 0.5 * (s_lo + s_hi)
        model.s.value = s_a_ser_usado
        model.write('modelo_debug.lp', io_options={'symbolic_solver_labels': True})

        print("="*18)
        res = opt.solve(model, load_solutions=False, tee=False)
        tc = res.solver.termination_condition
        print(f"Modelo Resolveu....Condição: ",tc)
        if tc == pyo.TerminationCondition.optimal:
            model.solutions.load_from(res)
            melhor_pesos = {list(model.nome_ativos.data())[a]: pyo.value(model.x[a]) for a in model.ativos}
            s_lo = model.s.value
            print(f"Valor encontrado para o Sharpe: ",s_lo)
            historico.append((s_a_ser_usado, 'viavel'))
        elif tc in (pyo.TerminationCondition.infeasible,
                    pyo.TerminationCondition.infeasibleOrUnbounded, pyo.TerminationCondition.unbounded,pyo.TerminationCondition.unknown):
            s_hi = s_a_ser_usado
            print(f"Atualização do S_HI : {s_hi}")
            print("="*28)
            historico.append((s_a_ser_usado, 'inviavel'))
        else:
            print(f'status inesperado em s={s_a_ser_usado:.4f}: {tc}')
            pass
    print("SAIU DO WHILE")
    print(f'Sharpe máximo (diário) ≈ {s_lo:.4f}  |  anualizado ≈ {s_lo*np.sqrt(252):.2f}')
    carteiras_anuais[int(ano_um)+1] = {
        'pesos':  melhor_pesos,
        'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano_um} -> {melhor_pesos}")
    print("+-="*30)
    if 'model' in locals():
        del model
        print('DELETADO DPS DO WHILE')
    else:
        print("Nao consta model")
        pass

=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2015
Nao consta model
Score Atualizado
Retornos atualizados
Excessos Atualizados
Sigmas Atualizados
-----
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2016
## UTILIZANDO SCORE DO ANO DE:  2015
## UTILIZANDO DADOS DE RETORNO DE:  2015
## UTILIZANDO EXCESSO DO ANO DE:  2015
## UTILIZANDO SIGMAS DO ANO DE:  2015
55
55
55
55
MODEL.S.VALUE =>  2
Começando o WHILE do ano 2015
Modelo Resolveu....Condição:  infeasible
Atualização do S_HI : 0.505
Modelo Resolveu....Condição:  optimal
Valor encontrado para o Sharpe:  0.2575
Modelo Resolveu....Condição:  optimal
Valor encontrado para o Sharpe:  0.38125
Modelo Resolveu....Condição:  optimal
Valor encontrado para o Sharpe:

In [37]:
carteiras_anuais

{2016: {'pesos': {'ABEV3': 1.666229104048239e-11,
   'ANIM3': 4.148272710760113e-12,
   'AXIA3': 6.97847860164749e-12,
   'AZZA3': 1.4840027169743187e-11,
   'B3SA3': 2.4970846301356728e-11,
   'BBSE3': 8.882189055538037e-12,
   'BEEF3': 6.896384160738179e-12,
   'BRAP4': 6.095866094019513e-12,
   'BRKM5': 9.658414730423709e-12,
   'CMIG4': 6.020618900076161e-12,
   'COGN3': 2.7948275067055035e-11,
   'CPFE3': 2.3683468504023925e-11,
   'CPLE3': 6.265702574757693e-12,
   'CSAN3': 6.298941481535231e-11,
   'CSMG3': 1.907394467779776e-11,
   'CSNA3': 1.847896927916323e-11,
   'CVCB3': 1.4987754195678583e-11,
   'CYRE3': 2.5046312078295854e-11,
   'DIRR3': 0.0781035005972123,
   'ECOR3': 9.821404594633597e-12,
   'EMBJ3': 1.0394645330118002e-11,
   'ENGI11': 0.0199999999286589,
   'EQTL3': 3.310921784304422e-11,
   'EZTC3': 1.2820816296376952e-11,
   'FLRY3': 0.13937759833604785,
   'GGBR4': 1.0101411658665795e-11,
   'GOAU4': 6.895278598406003e-12,
   'HYPE3': 0.1476037458477076,
   'ISA

In [34]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])+1
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        if v >= vb_peso_minimo and v <= vb_peso_maximo:
            linhas.append({'ano': an, 'ativo': k, 'peso': round(v, 4)})

df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


In [39]:
df_portfolios[df_portfolios['ano'] == 2017].sort_values('peso', ascending=False)['peso'].sum()

np.float64(1.0)

In [36]:
df_portfolios.to_csv('carteiras_mf.csv')
